In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# ---------------------------------
# Block 0: Google Earth Engine Data Extraction
# ---------------------------------
# PURPOSE:
# This script is a ONE-TIME data extraction task. It will:
# 1. Read your main training file to get a list of unique site locations and years.
# 2. For each unique site-year, query Google Earth Engine (GEE) for the 64-dimensional satellite embedding.
# 3. Save the results to a new CSV file in your Google Drive.
#
# NOTE: This script can take a significant amount of time to run, as it makes many individual requests to GEE.
#       Run it once, and then you can use the generated CSV file for all your future modeling work.
# ---------------------------------

import ee
import pandas as pd
import time

# --- 1. Authenticate and Initialize GEE ---
# IMPORTANT: You must specify a Google Cloud Project ID that has the Earth Engine API enabled.
# 1. Go to https://console.cloud.google.com/ to find your project ID.
# 2. Make sure the "Earth Engine API" is enabled for that project.
# 3. Paste your project ID in the variable below.
GCP_PROJECT_ID = 'abruptthawmapping' # <-- UPDATE THIS

print("Authenticating and initializing Google Earth Engine...")
try:
    # Authenticate first. This will trigger a pop-up authentication flow.
    ee.Authenticate()
    # Initialize with the specified project.
    ee.Initialize(project=GCP_PROJECT_ID)
    print("GEE initialized successfully.")
except Exception as e:
    print("\n--- ERROR ---")
    print("GEE Initialization Failed. Please ensure that:")
    print(f"1. You have updated the 'GCP_PROJECT_ID' variable in the script with a valid project ID.")
    print(f"2. The Google Earth Engine API is enabled for the project '{GCP_PROJECT_ID}'.")
    print(f"3. You have completed the authentication pop-up flow.")
    print(f"Original error: {e}")
    raise


# --- 2. Define File Paths ---
# Path to your main training data file
input_training_data_path = '/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_data_v5.csv'

# Path where the new satellite embedding CSV will be saved
output_embeddings_path = '/explore/nobackup/people/spotter5/anna_v/v2/satellite_embeddings.csv'


# --- 3. Load Source Data and Get Unique Site-Years ---
print(f"\nLoading source data from: {input_training_data_path}")
try:
    df_source = pd.read_csv(input_training_data_path, usecols=['site_name', 'latitude', 'longitude', 'year'])
    # Get a list of unique combinations of site, coordinates, and year
    unique_site_years = df_source.drop_duplicates(subset=['site_name', 'year']).reset_index(drop=True)
    print(f"Found {len(unique_site_years)} unique site-year combinations to process.")
except FileNotFoundError:
    print(f"ERROR: Source file not found at {input_training_data_path}. Please update the path.")
    raise


# --- 4. GEE Data Extraction Loop ---
# Load the GEE Image Collection for Satellite Embeddings
se_collection = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

results_list = []
total_sites = len(unique_site_years)

print("\nStarting GEE data extraction... This may take a while.")

for index, row in unique_site_years.iterrows():
    try:
        # Get site info
        site_name = row['site_name']
        original_year = int(row['year'])
        lat = row['latitude']
        lon = row['longitude']

        # --- MODIFICATION START ---
        # If the year is before 2017, use 2017 as a proxy for the query.
        # Otherwise, use the original year.
        query_year = 2017 if original_year < 2017 else original_year
        # --- MODIFICATION END ---

        # Define the point of interest and the date range using the query_year
        point = ee.Geometry.Point(lon, lat)
        start_date = f'{query_year}-01-01'
        end_date = f'{query_year+1}-01-01'

        # Filter the collection to get the specific image for the year and location
        image = se_collection.filterBounds(point).filterDate(start_date, end_date).first()

        # Check if an image was found
        if image:
            # Use reduceRegion to extract the pixel values (the 64 embedding dimensions)
            embedding_dict = image.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=10  # Native resolution of the dataset
            ).getInfo()

            # Prepare the result dictionary, crucially saving the ORIGINAL year
            # for correct merging later.
            result = {
                'site_name': site_name,
                'year': original_year
            }
            # Rename bands from 'A00', 'A01' ... to 'SE_0', 'SE_1' ...
            for i in range(64):
                band_name = f'A{i:02d}'
                new_name = f'SE_{i}'
                result[new_name] = embedding_dict.get(band_name)

            results_list.append(result)

        else:
            print(f"  - WARNING: No image found for {site_name} in query year {query_year} (original year: {original_year}).")

        # Print progress
        if (index + 1) % 10 == 0:
            print(f"  Processed {index + 1} / {total_sites} site-years...")

    except Exception as e:
        print(f"  - ERROR processing {site_name} for year {original_year}: {e}")
        # Add a small delay to avoid overwhelming the server in case of errors
        time.sleep(5)

print(f"\nExtraction complete. Successfully retrieved data for {len(results_list)} site-years.")


# --- 5. Create DataFrame and Save to CSV ---
if results_list:
    print("\nCreating DataFrame from results...")
    df_embeddings = pd.DataFrame(results_list)

    print(f"Saving embeddings to: {output_embeddings_path}")
    df_embeddings.to_csv(output_embeddings_path, index=False)
    print("File saved successfully.")

    print("\n--- Sample of the saved data ---")
    print(df_embeddings.head())
else:
    print("\nNo data was extracted. The output file was not created.")



Authenticating and initializing Google Earth Engine...
GEE initialized successfully.

Loading source data from: /content/drive/MyDrive/Flux_upscaling/v2_model_training_02Oct25.csv
Found 4486 unique site-year combinations to process.

Starting GEE data extraction... This may take a while.
  Processed 10 / 4486 site-years...
  Processed 20 / 4486 site-years...
  Processed 30 / 4486 site-years...
  Processed 40 / 4486 site-years...
  Processed 50 / 4486 site-years...
  Processed 60 / 4486 site-years...
  Processed 70 / 4486 site-years...
  Processed 80 / 4486 site-years...
  Processed 90 / 4486 site-years...
  Processed 100 / 4486 site-years...
  Processed 110 / 4486 site-years...
  Processed 120 / 4486 site-years...
  Processed 130 / 4486 site-years...
  Processed 140 / 4486 site-years...
  Processed 150 / 4486 site-years...
  - ERROR processing Lake Hazen, Ellesmere Island for year 2000: Image.reduceRegion: Parameter 'image' is required and may not be null.
  Processed 160 / 4486 site-y

  Processed 590 / 4486 site-years...
  Processed 600 / 4486 site-years...
  Processed 610 / 4486 site-years...
  Processed 620 / 4486 site-years...
  Processed 630 / 4486 site-years...
  Processed 640 / 4486 site-years...
  Processed 650 / 4486 site-years...
  Processed 660 / 4486 site-years...
  Processed 670 / 4486 site-years...
  Processed 680 / 4486 site-years...
  Processed 690 / 4486 site-years...
  - ERROR processing Lake Hazen, Ellesmere Island for year 2003: Image.reduceRegion: Parameter 'image' is required and may not be null.
  Processed 700 / 4486 site-years...
  Processed 710 / 4486 site-years...
  Processed 720 / 4486 site-years...
  Processed 730 / 4486 site-years...
  Processed 740 / 4486 site-years...
  Processed 750 / 4486 site-years...
  Processed 760 / 4486 site-years...
  Processed 770 / 4486 site-years...
  Processed 780 / 4486 site-years...
  Processed 790 / 4486 site-years...
  Processed 800 / 4486 site-years...
  Processed 810 / 4486 site-years...
  Processed 8